# Delta table Query util
DuckDB natively supports reading Delta tables using `delta_scan()`.

In [1]:
import os
import certifi
import duckdb
from electricity_maps.config import get_settings

os.environ["AWS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()


get_settings.cache_clear()
settings = get_settings()
DATA_DIR = settings.data_dir

In [2]:
import re

_con = None

def get_duckdb_connection():
    global _con
    if _con is not None:
        return _con

    _con = duckdb.connect()
    _con.execute("INSTALL delta;")
    _con.execute("LOAD delta;")

    if str(DATA_DIR).startswith('s3://'):
        _con.execute("INSTALL httpfs;")
        _con.execute("LOAD httpfs;")


        _con.execute(
            "CREATE OR REPLACE SECRET emaps_s3 (TYPE S3, KEY_ID ?, SECRET ?, REGION ?)",
            [
                settings.aws_access_key_id,
                settings.aws_secret_access_key,
                settings.aws_region or "ap-south-1",
            ],
        )

    return _con

def print_sql_df(sql_query):
    con = get_duckdb_connection()
    
    def replace_table(match):
        keyword = match.group(1)
        schema = match.group(2)
        table = match.group(3)
        
        if schema == "bronze":
            return f"{keyword} read_parquet('{DATA_DIR}/{schema}/{table}/**/*.parquet')"
        else:
            return f"{keyword} delta_scan('{DATA_DIR}/{schema}/{table}')"

    parsed_query = re.sub(
        r'(?i)(FROM|JOIN)\s+([a-zA-Z0-9_]+)\.([a-zA-Z0-9_]+)', 
        replace_table, 
        sql_query
    )
    df = con.execute(parsed_query).df()
    display(df)

In [3]:
print("Bronze Daily Flows Summary:")
print_sql_df('select * from bronze.electricity_flows limit 10')

Bronze Daily Flows Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777122094292,2026-04-25 18:31:34.292375+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-24 18:30:00+05:30,2026-04-25 18:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [4]:
print("Bronze Daily Mix Summary:")
print_sql_df('select * from bronze.electricity_mix limit 10')

Bronze Daily Mix Summary:


,process_ts,ingestion_timestamp,source_url,zone,range_start,range_end,raw_json,record_count,day,month,year
0,1777122094292,2026-04-25 18:31:34.292375+05:30,https://api.electricitymaps.com/v4/electricity...,FR,2026-04-24 18:30:00+05:30,2026-04-25 18:30:00+05:30,"{""zone"": ""FR"", ""temporalGranularity"": ""hourly""...",24,25,04,2026


In [5]:
print("Silver :")
print_sql_df('select * from silver.electricity_mix limit 10')

Silver :


,process_ts,zone,datetime,updated_at,is_estimated,estimation_method,nuclear_mw,geothermal_mw,biomass_mw,coal_mw,...,unknown_mw,hydro_storage_charge_mw,hydro_storage_discharge_mw,battery_storage_charge_mw,battery_storage_discharge_mw,flow_exports_mw,flow_imports_mw,year,month,day
0,1777122124981,FR,2026-04-24 18:30:00+05:30,2026-04-24 20:25:22.767000+05:30,True,SANDBOX_MODE_DATA,26839.733,NaN,912.339,0.0,...,NaN,1859.232,NaN,1.798,NaN,11394.0,0.0,2026,4,24
1,1777122124981,FR,2026-04-24 19:30:00+05:30,2026-04-24 22:39:44.546000+05:30,True,SANDBOX_MODE_DATA,38477.191,NaN,1400.147,0.0,...,NaN,1880.148,NaN,15.746,NaN,10358.0,0.0,2026,4,24
2,1777122124981,FR,2026-04-24 20:30:00+05:30,2026-04-24 22:40:58.002000+05:30,True,SANDBOX_MODE_DATA,32267.469,NaN,1046.077,0.0,...,NaN,1423.456,NaN,2.647,NaN,14758.0,0.0,2026,4,24
3,1777122124981,FR,2026-04-24 21:30:00+05:30,2026-04-24 23:42:55.823000+05:30,True,SANDBOX_MODE_DATA,35737.651,NaN,1110.081,0.0,...,NaN,248.792,NaN,4.393,NaN,19567.0,0.0,2026,4,24
4,1777122124981,FR,2026-04-24 22:30:00+05:30,2026-04-25 00:25:00.889000+05:30,True,SANDBOX_MODE_DATA,50577.859,NaN,1052.974,0.0,...,NaN,NaN,1651.566,NaN,3.972,16252.0,0.0,2026,4,24
5,1777122124981,FR,2026-04-24 23:30:00+05:30,2026-04-25 01:40:10.039000+05:30,True,SANDBOX_MODE_DATA,47691.042,NaN,1363.822,0.0,...,NaN,NaN,2957.200,NaN,119.522,16601.0,0.0,2026,4,24
6,1777122124981,FR,2026-04-25 00:30:00+05:30,2026-04-25 02:54:54.199000+05:30,True,SANDBOX_MODE_DATA,53733.346,NaN,1266.234,0.0,...,NaN,NaN,3658.718,NaN,165.202,16524.0,0.0,2026,4,24
7,1777122124981,FR,2026-04-25 01:30:00+05:30,2026-04-25 03:40:05.174000+05:30,True,SANDBOX_MODE_DATA,45460.254,NaN,1289.552,0.0,...,NaN,NaN,4256.418,18.185,NaN,20688.0,0.0,2026,4,24
8,1777122124981,FR,2026-04-25 02:30:00+05:30,2026-04-25 04:54:57.101000+05:30,True,SANDBOX_MODE_DATA,43035.688,NaN,1300.527,0.0,...,NaN,NaN,4009.250,0.960,NaN,17179.0,0.0,2026,4,24
9,1777122124981,FR,2026-04-25 03:30:00+05:30,2026-04-25 06:10:30.076000+05:30,True,SANDBOX_MODE_DATA,50165.579,NaN,1413.917,0.0,...,NaN,NaN,3917.475,NaN,3.495,22061.0,0.0,2026,4,24


In [6]:
print("Gold Daily Exports:")
print_sql_df('select * from gold.daily_exports limit 10')

Gold Daily Exports:


,process_ts,zone,zone_name,destination_zone,destination_zone_name,date,export_mwh,hours_covered,year,month
0,1777122140971,FR,France,LU,LU,2026-04-25,2522.0,13,2026,4
1,1777122140971,FR,France,IT-NO,Italy North,2026-04-25,14530.0,13,2026,4
2,1777122140971,FR,France,IT-NO,Italy North,2026-04-24,39519.0,11,2026,4
3,1777122140971,FR,France,GB,Great Britain,2026-04-25,37722.0,13,2026,4
4,1777122140971,FR,France,LU,LU,2026-04-24,1926.0,11,2026,4
5,1777122140971,FR,France,GB,Great Britain,2026-04-24,33810.0,11,2026,4


In [7]:
print("\nGold Daily Imports:")
print_sql_df("SELECT * FROM gold.daily_imports LIMIT 10")


Gold Daily Imports:


,process_ts,zone,zone_name,source_zone,source_zone_name,date,import_mwh,hours_covered,year,month
0,1777122140971,FR,France,ES,Spain,2026-04-24,26461.0,11,2026,4
1,1777122140971,FR,France,ES,Spain,2026-04-25,36571.0,13,2026,4
2,1777122140971,FR,France,CH,Switzerland,2026-04-24,15070.0,11,2026,4
3,1777122140971,FR,France,BE,Belgium,2026-04-24,36066.0,11,2026,4
4,1777122140971,FR,France,DE,Germany,2026-04-24,22955.0,11,2026,4
5,1777122140971,FR,France,DE,Germany,2026-04-25,12579.0,13,2026,4
6,1777122140971,FR,France,BE,Belgium,2026-04-25,37149.0,13,2026,4
7,1777122140971,FR,France,CH,Switzerland,2026-04-25,4541.0,13,2026,4


In [8]:
print("\nGold Daily Mix:")
print_sql_df("SELECT * FROM gold.daily_electricity_mix LIMIT 10")


Gold Daily Mix:


,process_ts,zone,zone_name,date,nuclear_pct,biomass_pct,wind_pct,solar_pct,hydro_pct,gas_pct,oil_pct,coal_pct,geothermal_pct,unknown_pct,total_production_mwh,fossil_free_avg_pct,renewable_avg_pct,hours_covered,year,month
0,1777122140971,FR,France,2026-04-24,72.495296,1.985227,7.588037,7.454164,9.780453,0.634279,0.062544,0.0,0.0,0.0,656375.916,99.303177,26.807881,11,2026,4
1,1777122140971,FR,France,2026-04-25,75.625031,2.216421,3.436989,9.398699,8.600121,0.653738,0.069001,0.0,0.0,0.0,696116.470,99.277261,23.652231,13,2026,4


In [9]:
print("\nPipeline State (el_state):")
print_sql_df("SELECT * FROM state.el_state ORDER BY process_ts DESC limit 10")


Pipeline State (el_state):


,process,process_ts,start_timestamp,end_timestamp,status,record_count,error_message
0,gold,1777122140971,2026-04-25 18:32:22.516493+05:30,2026-04-25 18:32:26.976657+05:30,R,16,None
1,silver,1777122124981,2026-04-25 18:32:06.420421+05:30,2026-04-25 18:32:10.078993+05:30,C,194,None
2,bronze,1777122094292,2026-04-25 18:31:34.292375+05:30,2026-04-25 18:31:37.369383+05:30,C,48,None
